In [1]:
import torch
import torch.nn as nn

from weight_mixture_linear_layer_config import DynamicLayerConfig, DynamicLayerOptions 
from utils.dynamic_layers import (VectorChoiceNaive, VectorChoiceEmbedding, MatrixChoice,
MatrixGenerator, VectorChoiceNaiveMixture, MatrixChoiceMixture, MatrixGeneratorMixture)

--- TEST: All dinamic linear layers 

In [7]:
config = DynamicLayerConfig()
# config.gather_frequency = False

def test(model_type):
    print('# ' + model_type)
    config.weight_choice_model = model_type 
    input = torch.randn(config.batch_size, config.d_input)
    print('Input size: ', input.shape)
    model = globals()[model_type](config)
    probs, idxs = model.forward_time(input)
    print('Output size: ', probs.shape)
    if hasattr(model, 'frequency'):
        print(model.frequency.frequency_data)
    print('\n' + '*'*50 + '\n')

for model_type in DynamicLayerOptions:
    test(model_type.value)

# VectorChoiceNaive
Input size:  torch.Size([128, 256])
forward_time executed in 0.1730 seconds
Output size:  torch.Size([128, 256, 64])
tensor([[3., 2., 8.,  ..., 2., 3., 4.],
        [1., 6., 4.,  ..., 4., 5., 4.],
        [1., 3., 6.,  ..., 1., 4., 3.],
        ...,
        [1., 6., 5.,  ..., 4., 6., 5.],
        [6., 6., 5.,  ..., 3., 3., 4.],
        [2., 3., 7.,  ..., 5., 3., 3.]])

**************************************************

# VectorChoiceEmbedding
Input size:  torch.Size([128, 256])
forward_time executed in 0.2176 seconds
Output size:  torch.Size([128, 256, 64])

**************************************************

# MatrixChoice
Input size:  torch.Size([128, 256])
forward_time executed in 0.0662 seconds
Output size:  torch.Size([128, 256, 64])
tensor([5, 5, 7, 2, 5, 4, 4, 3, 7, 4, 3, 1, 5, 4, 1, 4, 1, 2, 3, 6, 1, 6, 1, 4,
        3, 4, 4, 6, 6, 5, 4, 8])

**************************************************

# MatrixGenerator
Input size:  torch.Size([128, 256])
forward_ti

TEST: Dynamic linear layers

In [ ]:
from enum import Enum
from torch.nn import functional as F

class MatrixChoiceMixture(DynamicLayers):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.noisy_gating = cfg.noisy_gating
        self.k = min(cfg.k, self.d_depth)
        self.mlp_router = cfg.mlp_router

        self.w_bank = nn.Parameter(torch.randn(self.d_depth, self.d_input, self.d_output), requires_grad=True)

        # TODO: Temporary inizialize the weights with xavier initiaziation but will
        # be initialzed at cluster level when done
        init.xavier_uniform_(self.w_bank)
        self.router.apply(xavier_init_weights)

    def forward(self, x, sample_topk=0):
        # Given an input of shape [batch_size, d_input] generate probabilities and their 
        # indices of shape [d_input, batch_size, topk]
        top_k_gates, top_k_indices = self.top_k_gating(x, sample_topk)

        # Select k weight matrices for each sample [batch_size * k, d_input, d_output]
        selected_weights = self.w_bank[top_k_indices.reshape(-1)]
        
        # Multiply each weight matrice by it's weight [batch_size * k, d_input, d_output] * [batch_size * k, 1, 1] 
        selected_weights = selected_weights * top_k_gates.reshape(-1, 1, 1) 
        
        # Reshape weights into [batch_size, k, d_input, d_output]
        selected_weights = selected_weights.reshape(self.batch_size, -1, self.d_input, self.d_output)
        
        # Sum the k dimension resulntin into [batch_size, d_input, d_output] 
        selected_weights = selected_weights.sum(dim=1)

        return selected_weights, None


class DynamicLayerTestActivation(Enum):
    activation_none = None
    activation_sigmoid = F.sigmoid
    activation_tanh = F.tanh
    activation_relu = F.relu
    activation_gelu = F.gelu
    activation_elu = F.elu


class DynamicLayerConfigNoise(Enum):
    noisy_gating = True
    sample_topk = True
    noisy_gating_and_sample_topk = True

config = DynamicLayerConfig()
config.gather_frequency = False

def testMatrixChoiceMixtureActivation():
    scores = {}
    for activation in DynamicLayerTestActivation:
        config.weight_choice_model = DynamicLayerOptions.matrix_generatior2.value
        model = WeightMixtureLayer(cfg=config)
        model.temp_activation = activation.value
        trainer = Trainer(max_epochs=20)
        trainer.fit(model, data)

In [4]:
x = torch.arange(4 * 5 * 7).reshape(4, 5, 7)
y = torch.randint(0, 5, (4, 2, 3))
print(x)
print(y)

tensor([[[  0,   1,   2,   3,   4,   5,   6],
         [  7,   8,   9,  10,  11,  12,  13],
         [ 14,  15,  16,  17,  18,  19,  20],
         [ 21,  22,  23,  24,  25,  26,  27],
         [ 28,  29,  30,  31,  32,  33,  34]],

        [[ 35,  36,  37,  38,  39,  40,  41],
         [ 42,  43,  44,  45,  46,  47,  48],
         [ 49,  50,  51,  52,  53,  54,  55],
         [ 56,  57,  58,  59,  60,  61,  62],
         [ 63,  64,  65,  66,  67,  68,  69]],

        [[ 70,  71,  72,  73,  74,  75,  76],
         [ 77,  78,  79,  80,  81,  82,  83],
         [ 84,  85,  86,  87,  88,  89,  90],
         [ 91,  92,  93,  94,  95,  96,  97],
         [ 98,  99, 100, 101, 102, 103, 104]],

        [[105, 106, 107, 108, 109, 110, 111],
         [112, 113, 114, 115, 116, 117, 118],
         [119, 120, 121, 122, 123, 124, 125],
         [126, 127, 128, 129, 130, 131, 132],
         [133, 134, 135, 136, 137, 138, 139]]])
tensor([[[4, 0, 0],
         [0, 4, 4]],

        [[2, 4, 1],
         [

In [5]:
batch_indices  = torch.arange(4).view(4, 1, 1).expand(-1, 2, 3)
print(batch_indices)
print(batch_indices.shape)

tensor([[[0, 0, 0],
         [0, 0, 0]],

        [[1, 1, 1],
         [1, 1, 1]],

        [[2, 2, 2],
         [2, 2, 2]],

        [[3, 3, 3],
         [3, 3, 3]]])
torch.Size([4, 2, 3])


In [6]:
z = torch.tensor(
  [[[10, 100, 1000],
    [2, 2, 2]],

    [[3, 3, 3],
     [4, 4, 4]],

    [[5, 5, 5],
     [6, 6, 6]],

    [[7, 7, 7],
     [8, 8, 8]]]
).unsqueeze(-1)

print(z.shape)

torch.Size([4, 2, 3, 1])


In [7]:
selected_vectors = x[batch_indices, y]
print(selected_vectors.shape)
print(selected_vectors)

torch.Size([4, 2, 3, 7])
tensor([[[[ 28,  29,  30,  31,  32,  33,  34],
          [  0,   1,   2,   3,   4,   5,   6],
          [  0,   1,   2,   3,   4,   5,   6]],

         [[  0,   1,   2,   3,   4,   5,   6],
          [ 28,  29,  30,  31,  32,  33,  34],
          [ 28,  29,  30,  31,  32,  33,  34]]],


        [[[ 49,  50,  51,  52,  53,  54,  55],
          [ 63,  64,  65,  66,  67,  68,  69],
          [ 42,  43,  44,  45,  46,  47,  48]],

         [[ 42,  43,  44,  45,  46,  47,  48],
          [ 56,  57,  58,  59,  60,  61,  62],
          [ 63,  64,  65,  66,  67,  68,  69]]],


        [[[ 98,  99, 100, 101, 102, 103, 104],
          [ 84,  85,  86,  87,  88,  89,  90],
          [ 84,  85,  86,  87,  88,  89,  90]],

         [[ 84,  85,  86,  87,  88,  89,  90],
          [ 70,  71,  72,  73,  74,  75,  76],
          [ 77,  78,  79,  80,  81,  82,  83]]],


        [[[126, 127, 128, 129, 130, 131, 132],
          [105, 106, 107, 108, 109, 110, 111],
          [133, 1

In [8]:
final = selected_vectors * z
print(final.shape)
print(final.sum(dim=-2).transpose(1,0).shape)
print(final)

torch.Size([4, 2, 3, 7])
torch.Size([2, 4, 7])
tensor([[[[ 280,  290,  300,  310,  320,  330,  340],
          [   0,  100,  200,  300,  400,  500,  600],
          [   0, 1000, 2000, 3000, 4000, 5000, 6000]],

         [[   0,    2,    4,    6,    8,   10,   12],
          [  56,   58,   60,   62,   64,   66,   68],
          [  56,   58,   60,   62,   64,   66,   68]]],


        [[[ 147,  150,  153,  156,  159,  162,  165],
          [ 189,  192,  195,  198,  201,  204,  207],
          [ 126,  129,  132,  135,  138,  141,  144]],

         [[ 168,  172,  176,  180,  184,  188,  192],
          [ 224,  228,  232,  236,  240,  244,  248],
          [ 252,  256,  260,  264,  268,  272,  276]]],


        [[[ 490,  495,  500,  505,  510,  515,  520],
          [ 420,  425,  430,  435,  440,  445,  450],
          [ 420,  425,  430,  435,  440,  445,  450]],

         [[ 504,  510,  516,  522,  528,  534,  540],
          [ 420,  426,  432,  438,  444,  450,  456],
          [ 462,  468

---

In [9]:
x = torch.arange(6 * 3 * 7).reshape(6, 3, 7)
y = torch.tensor([[4, 3],
                  [1, 4]])
print(x)
print(y)
print(y.reshape(-1, 1))

tensor([[[  0,   1,   2,   3,   4,   5,   6],
         [  7,   8,   9,  10,  11,  12,  13],
         [ 14,  15,  16,  17,  18,  19,  20]],

        [[ 21,  22,  23,  24,  25,  26,  27],
         [ 28,  29,  30,  31,  32,  33,  34],
         [ 35,  36,  37,  38,  39,  40,  41]],

        [[ 42,  43,  44,  45,  46,  47,  48],
         [ 49,  50,  51,  52,  53,  54,  55],
         [ 56,  57,  58,  59,  60,  61,  62]],

        [[ 63,  64,  65,  66,  67,  68,  69],
         [ 70,  71,  72,  73,  74,  75,  76],
         [ 77,  78,  79,  80,  81,  82,  83]],

        [[ 84,  85,  86,  87,  88,  89,  90],
         [ 91,  92,  93,  94,  95,  96,  97],
         [ 98,  99, 100, 101, 102, 103, 104]],

        [[105, 106, 107, 108, 109, 110, 111],
         [112, 113, 114, 115, 116, 117, 118],
         [119, 120, 121, 122, 123, 124, 125]]])
tensor([[4, 3],
        [1, 4]])
tensor([[4],
        [3],
        [1],
        [4]])


In [10]:
z = torch.arange(2).view(-1, 1, 1).expand(-1, 2, 1)
print(z)

tensor([[[0],
         [0]],

        [[1],
         [1]]])


In [11]:
g = torch.tensor([[0, 1],
                  [2, 3]])
#print(x[y].shape)
print(x[y.reshape(-1)].shape)
print(x[y.reshape(-1)].reshape(2, -1, 3, 7).shape)
print(x[y.reshape(-1)])
print(y.reshape(-1,1,1))
print(x[y.reshape(-1)] * g.reshape(-1, 1, 1))
print(x[y.reshape(-1)].reshape(2, -1, 3, 7))
print(torch.sum(x[y.reshape(-1)].reshape(2, -1, 3, 7), dim=1).shape)

#print(x[z, y])
"""
tensor([[4],
        [3],
        [1],
        [4]])
"""

torch.Size([4, 3, 7])
torch.Size([2, 2, 3, 7])


tensor([[[ 84,  85,  86,  87,  88,  89,  90],
         [ 91,  92,  93,  94,  95,  96,  97],
         [ 98,  99, 100, 101, 102, 103, 104]],

        [[ 63,  64,  65,  66,  67,  68,  69],
         [ 70,  71,  72,  73,  74,  75,  76],
         [ 77,  78,  79,  80,  81,  82,  83]],

        [[ 21,  22,  23,  24,  25,  26,  27],
         [ 28,  29,  30,  31,  32,  33,  34],
         [ 35,  36,  37,  38,  39,  40,  41]],

        [[ 84,  85,  86,  87,  88,  89,  90],
         [ 91,  92,  93,  94,  95,  96,  97],
         [ 98,  99, 100, 101, 102, 103, 104]]])
tensor([[[4]],

        [[3]],

        [[1]],

        [[4]]])
tensor([[[  0,   0,   0,   0,   0,   0,   0],
         [  0,   0,   0,   0,   0,   0,   0],
         [  0,   0,   0,   0,   0,   0,   0]],

        [[ 63,  64,  65,  66,  67,  68,  69],
         [ 70,  71,  72,  73,  74,  75,  76],
         [ 77,  78,  79,  80,  81,  82,  83]],

        [[ 42,  44,  46,  48,  50,  52,  54],
         [ 56,  58,  60,  62,  64,  66,  68],
    

'\ntensor([[4],\n        [3],\n        [1],\n        [4]])\n'

----

In [2]:
x = torch.arange(6 * 3 * 7).reshape(6, 3, 7)
y = torch.tensor([[4, 3],
                  [1, 4]])

y = y.reshape(-1)
print(x)
print(y)

tensor([[[  0,   1,   2,   3,   4,   5,   6],
         [  7,   8,   9,  10,  11,  12,  13],
         [ 14,  15,  16,  17,  18,  19,  20]],

        [[ 21,  22,  23,  24,  25,  26,  27],
         [ 28,  29,  30,  31,  32,  33,  34],
         [ 35,  36,  37,  38,  39,  40,  41]],

        [[ 42,  43,  44,  45,  46,  47,  48],
         [ 49,  50,  51,  52,  53,  54,  55],
         [ 56,  57,  58,  59,  60,  61,  62]],

        [[ 63,  64,  65,  66,  67,  68,  69],
         [ 70,  71,  72,  73,  74,  75,  76],
         [ 77,  78,  79,  80,  81,  82,  83]],

        [[ 84,  85,  86,  87,  88,  89,  90],
         [ 91,  92,  93,  94,  95,  96,  97],
         [ 98,  99, 100, 101, 102, 103, 104]],

        [[105, 106, 107, 108, 109, 110, 111],
         [112, 113, 114, 115, 116, 117, 118],
         [119, 120, 121, 122, 123, 124, 125]]])
tensor([4, 3, 1, 4])


In [4]:
selected_weight = x[y]
print(selected_weight.shape)
print(selected_weight)
print()
selected_weight = selected_weight.reshape(2, -1, 3, 7)
print(selected_weight.shape)
print(selected_weight)


torch.Size([4, 3, 7])
tensor([[[ 84,  85,  86,  87,  88,  89,  90],
         [ 91,  92,  93,  94,  95,  96,  97],
         [ 98,  99, 100, 101, 102, 103, 104]],

        [[ 63,  64,  65,  66,  67,  68,  69],
         [ 70,  71,  72,  73,  74,  75,  76],
         [ 77,  78,  79,  80,  81,  82,  83]],

        [[ 21,  22,  23,  24,  25,  26,  27],
         [ 28,  29,  30,  31,  32,  33,  34],
         [ 35,  36,  37,  38,  39,  40,  41]],

        [[ 84,  85,  86,  87,  88,  89,  90],
         [ 91,  92,  93,  94,  95,  96,  97],
         [ 98,  99, 100, 101, 102, 103, 104]]])

torch.Size([2, 2, 3, 7])
tensor([[[[ 84,  85,  86,  87,  88,  89,  90],
          [ 91,  92,  93,  94,  95,  96,  97],
          [ 98,  99, 100, 101, 102, 103, 104]],

         [[ 63,  64,  65,  66,  67,  68,  69],
          [ 70,  71,  72,  73,  74,  75,  76],
          [ 77,  78,  79,  80,  81,  82,  83]]],


        [[[ 21,  22,  23,  24,  25,  26,  27],
          [ 28,  29,  30,  31,  32,  33,  34],
          

In [11]:
# [2, 3] * [2, 2, 3, 7] 
input = torch.tensor([[2., 2., 2.],
                      [3., 3., 3.]]).float()
print(torch.matmul(input[0], selected_weight[0].float()))
print(torch.einsum('bi,bkij->bkj', input, selected_weight.float()))
print(torch.einsum('bi,bkij->bkj', input, selected_weight.float()).shape)

tensor([[546., 552., 558., 564., 570., 576., 582.],
        [420., 426., 432., 438., 444., 450., 456.]])
tensor([[[546., 552., 558., 564., 570., 576., 582.],
         [420., 426., 432., 438., 444., 450., 456.]],

        [[252., 261., 270., 279., 288., 297., 306.],
         [819., 828., 837., 846., 855., 864., 873.]]])
torch.Size([2, 2, 7])


In [32]:
xx = torch.einsum('bi,bkij->bkj', input, selected_weight.float())
yy = torch.einsum('bi,bkij->bkj', input, selected_weight.float())
zz = (torch.einsum("bij,bik->bijk", xx, yy) / 1000).int()
print(zz)
print(zz.shape)
print()
print(y)
print(y.reshape(2, -1, 1, 1))
print(zz * y.reshape(2, -1, 1, 1).float())

tensor([[[[298, 301, 304, 307, 311, 314, 317],
          [301, 304, 308, 311, 314, 317, 321],
          [304, 308, 311, 314, 318, 321, 324],
          [307, 311, 314, 318, 321, 324, 328],
          [311, 314, 318, 321, 324, 328, 331],
          [314, 317, 321, 324, 328, 331, 335],
          [317, 321, 324, 328, 331, 335, 338]],

         [[176, 178, 181, 183, 186, 189, 191],
          [178, 181, 184, 186, 189, 191, 194],
          [181, 184, 186, 189, 191, 194, 196],
          [183, 186, 189, 191, 194, 197, 199],
          [186, 189, 191, 194, 197, 199, 202],
          [189, 191, 194, 197, 199, 202, 205],
          [191, 194, 196, 199, 202, 205, 207]]],


        [[[ 63,  65,  68,  70,  72,  74,  77],
          [ 65,  68,  70,  72,  75,  77,  79],
          [ 68,  70,  72,  75,  77,  80,  82],
          [ 70,  72,  75,  77,  80,  82,  85],
          [ 72,  75,  77,  80,  82,  85,  88],
          [ 74,  77,  80,  82,  85,  88,  90],
          [ 77,  79,  82,  85,  88,  90,  93]],

     

---- TEST: WeightMixtureLayer

In [ ]:
from weight_mixture_linear_layer import WeightMixtureLayer
from weight_mixture_linear_layer_config import WeightChoiceVectorOptions, WeightChoiceMatrixOptions

def testWeightMixtureLayer():
    config = WeightMixtureLinearLayerConfig()
    print(config.weight_choice_model)
    config.gather_frequency = False
    config.weight_matrix_choice = False
    #config.weight_choice_model = WeightChoiceVectorOptions.vector_choice_naive.value
    config.weight_choice_model = WeightChoiceVectorOptions.vector_choice_naive.value
    input = torch.randn(config.batch_size, config.d_input)
    print('Input size: ', input.shape)
    model = WeightMixtureLayer(config)
    probs = model(input)
    print('Output.shape', probs.shape)
    print('-'*10)

testWeightMixtureLayer()

ImportError: attempted relative import with no known parent package

----

In [ ]:
x = torch.arange(20).reshape(2,10)
print(x)
x = x.repeat_interleave(3,dim=0)
print(x)
print(x.reshape(2, 3, 10))

In [ ]:
import torch
import torch.nn as nn
from dataclasses import dataclass, field, fields
import inspect
import torchvision
from torchvision import transforms
from matplotlib import pyplot as plt


import numpy as np
import torch
import torchvision
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils import data
from torchvision import transforms
from IPython import display

import collections

from base.utils import Module, Trainer
from base.models import Classifier
from base.datasets import FashionMNIST 

In [ ]:
class SoftmaxRegression(Classifier): 
    """The softmax regression model."""
    def __init__(self, num_outputs, lr):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(nn.Flatten(), nn.LazyLinear(num_outputs))

    def forward(self, X):
        return self.net(X)

data = FashionMNIST(batch_size=256)
model = SoftmaxRegression(num_outputs=10, lr=0.1)
trainer = Trainer(max_epochs=10)
#trainer.fit(model, data)

In [ ]:
@dataclass
class LinearLayerConfig():
    batch_size: int = field(default=32, metadata={"help": "Input dimension"})
    d_input: int = field(default=7, metadata={"help": "Input dimension"})
    d_output: int = field(default=5, metadata={"help": "Output dimension"})
    d_depth: int = field(default=10, metadata={"help": "Number of dimensions added to the tensor from which the weights will be chosen to form a new layer"})

---- WeightsMixtureDiscreteChoice

---- WeightsMixtureDiscreteChoiceEmbedding

In [ ]:
class WeightsMixtureDiscreteChoiceEmbedding(nn.Module):
    """
    This does not train. Weight's do not update when training this. This needs to be investigated. Ah
    and for some reason the output of the switch linear layer blows up to some very large numbers.
    Probably some better initialization for the weights would help.

    Idea is to have a version with less parameters than `SparseSwitch`

    How it works:
    - have a set of embeddings for each weight vector to be chosen of shape [d_input, d_switch_embedding]
    - given an input of shape [1, d_input] it will be repeated into a shape of [d_input, d_input]
    - concatenate the repeated input with embeddings into a shape of [d_input, d_input + d_switch_embedding]
    - the concatenation is processed a 2 layer linear layer that will generate scores for 
      each weight vector [d_input, d_depth]
    - calculate the softmax over the scores and store the largest probability and it's index both of shape
    """
    def __init__(self, cfg):
        self.batch_size = cfg.batch_size
        self.d_input = cfg.d_input
        self.d_depth = cfg.d_depth
        self.d_switch_embeding = cfg.d_switch_embeding

        self.switch = nn.Sequential(
            nn.Linear(self.d_input + self.d_switch_embeding, self.d_input + self.d_switch_embeding),
            nn.ReLU(),
            nn.Linear(self.d_input + self.d_switch_embeding, self.d_depth))

        self.switch_embedding_weights = nn.Parameter(torch.randn(self.d_input, self.d_switch_embeding))

    def forward(self, x):
        # Repeat each input vector from [batch_size, d_input] into [batch_size * d_input, d_input]
        repeated_input = x.repeat_interleave(self.d_input, dim=0) 

        # Repeat the embeddings of each connection from [d_input, d_switch_embeding] into [batch_size * d_input, d_switch_embeding]
        repeated_switch_embedding_weight = self.switch_embedding_weights.repeat(self.batch_size, 1)

        # Concatenate the inputs and repeated embeddings into a shape of [batch_size * d_input, d_input + d_switch_embedding]
        x = torch.cat((repeated_input, repeated_switch_embedding_weight), dim=-1)

        # Calculate the scores for each weight vector of shape [batch_size * d_input, d_depth]
        x = self.switch(x).reshape(self.batch_size, self.d_input, -1)
        
        # Get the highest probability and indexes of the highest probabilities
        probs, idxs = torch.max(F.softmax(x ,dim=-1), dim=-1)

        return probs, idxs

---- WeightMixtureLayer

In [ ]:
class WeightMixtureLayer(Module):
    def __init__(self, cfg):
        """
        This class creates a sparse linear layer.        
        """
        super().__init__()
        self.d_input = cfg.d_input
        self.d_output = cfg.d_output
        self.d_depth = cfg.d_depth
        self.switch_class = cfg.switch_model
        self.gather_frequency = cfg.gather_frequency
        self.switch = globals()[self.switch_class](cfg)

        self.w = nn.Parameter(torch.randn(self.d_input, self.d_depth, self.d_output), requires_grad=True)
        self.b = nn.Parameter(torch.zeros(1, self.d_output), requires_grad=True)
        if self.gather_frequency:
            self.frequency = WeightMixtureDepthUsageData(self.d_input, self.d_depth)

    def forward(self, x):
        """
        Single input example
        """
        # Predict a (1, 64) tensor where each number is a score for one out of self.depth
        # of vectors that can be selected for each row
        probs, idxs = self.switch.predict_weights(x)
        if self.gather_frequency:
            self.frequency.update_frequecy(idxs)
        # Select the specific weights for the current input
        selected_weights = self.w[torch.arange(self.w.shape[0]).unsqueeze(-1), idxs].transpose(1,0)

        # Pass each input through it's selected weight layer but using the same biases
        output = (torch.matmul(x.unsqueeze(1), selected_weights).squeeze(1) + self.b) 

        # Scale each output vector by the mean of probabilities of it's selected weights
        output = output * probs.mean(dim=0).unsqueeze(-1)

        return output



In [ ]:
class FeedForwardSparseLayer(Module):
    def __init__(self, cfg):
        """
        This class creates a sparse linear layer.
        """
        super().__init__()
        self.d_input = cfg.d_input
        self.d_output = cfg.d_output
        self.d_depth = cfg.d_depth
        self.weight_switch_softmax_check = True
        
  
        if self.weight_switch_softmax_check: 
            self.weight_switch_softmax_init()
        else:
            self.weight_switch_sigmoid_init()

        self.weights = nn.Parameter(torch.randn(self.d_input, self.d_depth, self.d_output), requires_grad=True)
        self.biases = nn.Parameter(torch.zeros(1, self.d_output), requires_grad=True)

        self.values = torch.tensor([0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85, 0.95])


    def forward(self, x):
        """
        Single input example
        """
        # Predict a (1, 64) tensor where each number is a score for one out of self.depth
        # of vectors that can be selected for each row
        if self.weight_switch_softmax_check:
            probs, idx = self.weight_switch_sofmtax_prediction(x)
        else:
            probs, idx = self.weight_switch_sigmoid_prediction(x)

        # Select the specific weights for the current input
        idx = idx.unsqueeze(-1).expand(-1, -1, self.d_output)
        selected_weights = torch.gather(self.weights, 1, idx).transpose(0,1)

        # Multiply the weights for the specific input
        output = (torch.matmul(x.unsqueeze(1), selected_weights).squeeze(1) + self.biases) *  probs.mean(dim=0).unsqueeze(-1)
        
        return output


    def weight_switch_sigmoid_init(self):
        self.switch = nn.Sequential(
            nn.Linear(self.d_input, self.d_input * 2),
            nn.ReLU(),
            nn.Linear(self.d_input * 2, self.d_output))
    
    def weight_switch_softmax_init(self):
        self.switch_weights = nn.Parameter(torch.randn(self.d_input, self.d_input, self.d_depth))
        self.switch_bias = nn.Parameter(torch.randn(self.d_input, 1, self.d_depth))

    def weight_switch_sigmoid_prediction(self, x):
        pass

    def weight_switch_sofmtax_prediction(self, x):
        """
        Uses softmax to generate a distribution for every weight vector to be selected

        Return:
            - probs: probabilities for each selected weight vector
            - idx: indices of the selected weight vectors
        
        Problems with this version:
            Number of parameters for prediction one weight matrix is very high (self.d_input ** 2 * self.d_depth).
            The reason you need this many parameters is that 1 input will be used to make all predictions for all
            weights of a layer.
        """
        # Pass the output above through a different weight matrix for each weight vector to be selected
        x = torch.matmul(x, self.switch_weights) + self.switch_bias
        
        # Calculate a softmax distribution for each weight vector 
        distrubtion = torch.softmax(x, dim=-1)
        
        # Calculate store the indices of the highest probabilites into a vector of shape (64x1) 
        probs, idx = torch.max(distrubtion, dim=-1)
        
        return probs, idx



def test():
    config = LinearLayerConfig()
    model = FeedForwardSparseLayer(cfg=config)
    print(model.weights.shape)
    print(model.biases.shape)
    input_test = torch.randn(2, 7)
    print('-'*20)
    output = model(input_test)
    print('-'*20)
    print(output)
    print(output.shape) 
test()

"""
What am i getting here ?
- i am supposed to ah probably i need to transpose the weights 
now to be 2x64x5 
"""

In [ ]:
batch_size = 2
num_features = 4
embedding_dim = 3
num_indices = 2

weights = torch.randn(batch_size, num_features, embedding_dim)
indices = torch.tensor([[1, 3], [0, 2]])  # 2D tensor of indices
print(weights)

# Add a singleton dimension to the end of indices
indices_expanded = indices.unsqueeze(-1).expand(-1, -1, embedding_dim)

print("indices_expanded:")
print(indices_expanded)

# Gather elements from weights using indices_expanded
selected_weights = torch.gather(weights, 1, indices_expanded)

print("selected_weights:")
print(selected_weights)

In [ ]:
"""

- put every method in their different files: done
- the only way to do it 

"""

In [ ]:
@dataclass
class LinearLayerConfig():
    batch_size: int = field(default=16, metadata={"help": "Number of samples in the batch"})
    d_input: int = field(default=768, metadata={"help": "Input dimension"})
    d_output: int = field(default=10, metadata={"help": "Output dimension"})
    d_depth: int = field(default=5, metadata={"help": "Number of dimensions added to the tensor from which the weights will be chosen to form a new layer"})
    d_switch_embeding: int = field(default=64, metadata={"help": "Output dimension"})
    switch_model: str = field(default='SparseSwitchEmbedding')


class SparseSwitchEmbedding(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.batch_size = cfg.batch_size
        self.d_input = cfg.d_input
        self.d_depth = cfg.d_depth
        self.d_switch_embeding = cfg.d_switch_embeding

        self.switch = nn.Sequential(
            nn.Linear(self.d_input + self.d_switch_embeding, self.d_input),
            nn.ReLU(),
            nn.Linear(self.d_input, self.d_depth)
        )
        self.switch_embedding_weights = torch.normal(0, 0.01, size=(self.d_input, self.d_switch_embeding), requires_grad=True)
        

    def forward(self, x):
        print('f'*20)
        # Repeat each input vector self.d_input times 
        repeated_input = x.unsqueeze(1).repeat(1, self.d_input, 1)
        print(repeated_input.shape)
        #tensor.unsqueeze(1).repeat(1, 3, 1)

        # Repeat the switch_embedding_weights 
        repeated_switch_embedding_weights =  self.switch_embedding_weights.repeat(self.batch_size, 1, 1)  # Shape (3, 3, 2)
        print(repeated_switch_embedding_weights.shape)
        # Concatenate inputs with shared embeddings this is done in order to make a prediction 
        # for each weight vector given an input
        x = torch.cat((repeated_input, repeated_switch_embedding_weights), dim=2).reshape(self.batch_size * self.d_input, -1)
        print(x.shape)
        x = self.switch(x)
        print(x.shape)

        x = x.reshape(self.batch_size, self.d_input, -1)
        print(x.shape)

        probs, idxs = torch.max(F.softmax(x, dim=-1), dim=-1)
        print(probs.shape)
        print('f'*20)
        return probs, idxs


class SparseSwitchEmbedding(nn.Module):
    """
    This does not train. Weight's do not update when training this. This needs to be investigated. Ah
    and for some reason the output of the switch linear layer blows up to some very large numbers.
    Probably some better initialization for the weights would help.

    Idea is to have a version with less parameters than `SparseSwitch`

    How it works:
    - have a set of embeddings for each weight vector to be chosen of shape [d_input, d_switch_embedding]
    - given an input of shape [1, d_input] it will be repeated into a shape of [d_input, d_input]
    - concatenate the repeated input with embeddings into a shape of [d_input, d_input + d_switch_embedding]
    - the concatenation is processed a 2 layer linear layer that will generate scores for 
      each weight vector [d_input, d_depth]
    - calculate the softmax over the scores and store the largest probability and it's index both of shape
    """
    def __init__(self, cfg):
        super().__init__()
        self.batch_size = cfg.batch_size
        self.d_input = cfg.d_input
        self.d_depth = cfg.d_depth
        self.d_switch_embeding = cfg.d_switch_embeding

        self.switch = nn.Sequential(
            nn.Linear(self.d_input + self.d_switch_embeding, self.d_input + self.d_switch_embeding),
            nn.ReLU(),
            nn.Linear(self.d_input + self.d_switch_embeding, self.d_depth))

        self.switch_embedding_weights = nn.Parameter(torch.randn(self.d_input, self.d_switch_embeding))

    def forward(self, x):
        # Repeat each input vector from [batch_size, d_input] into [batch_size * d_input, d_input]
        repeated_input = x.repeat_interleave(self.d_input, dim=0) 

        # Repeat the embeddings of each connection from [d_input, d_switch_embeding] into [batch_size * d_input, d_switch_embeding]
        repeated_switch_embedding_weight = self.switch_embedding_weights.repeat(self.batch_size, 1)

        # Concatenate the inputs and repeated embeddings into a shape of [batch_size * d_input, d_input + d_switch_embedding]
        x = torch.cat((repeated_input, repeated_switch_embedding_weight), dim=-1)

        # Calculate the scores for each weight vector of shape [batch_size * d_input, d_depth]
        x = self.switch(x).reshape(self.batch_size, self.d_input, -1)
        
        # Get the highest probability and indexes of the highest probabilities
        probs, idxs = torch.max(F.softmax(x ,dim=-1), dim=-1)

        return probs, idxs



class SparseFeedFowradLayer(Module):
    def __init__(self, cfg):
        """
        This class creates a sparse linear layer.        
        """
        super().__init__()
        self.d_input = cfg.d_input
        self.d_output = cfg.d_output
        self.d_depth = cfg.d_depth
        self.switch = SparseSwitchEmbedding(cfg=cfg)
        #self.switch = globals()[self.switch_class](cfg)

        # [768, 5, 10]
        self.w = nn.Parameter(torch.randn(self.d_input, self.d_depth, self.d_output), requires_grad=True)
        self.b = nn.Parameter(torch.zeros(1, self.d_output), requires_grad=True)


    def forward(self, x):
        """
        Single input example
        """
        # Predict a (1, 64) tensor where each number is a score for one out of self.depth
        # of vectors that can be selected for each row
        probs, idxs = self.switch(x)
        print(idxs.shape)
        print(self.w.shape)
        print(idxs.T.shape)
        # Select the specific weights for the current input
        selected_weights = self.w[torch.arange(self.w.shape[0]).unsqueeze(-1), idxs.T].transpose(1,0)
        print(selected_weights.shape)
        print('bla')
        # Pass each input through it's selected weight layer but using the same biases
        output = (torch.matmul(x.unsqueeze(1), selected_weights).squeeze(1) + self.b)
     
        # Scale each output vector by the mean of probabilities of it's selected weights
        output = output * probs.mean(dim=-1, keepdims=True)

        return output
    
    
def test():
    config = LinearLayerConfig()
    model = SparseFeedFowradLayer(cfg=config)
    input_test = torch.randn(LinearLayerConfig.batch_size, LinearLayerConfig.d_input)
    print('-'*20)
    output = model(input_test)
    print('-'*20)
    #print(output)
    print(output.shape) 
test()

In [ ]:
x = torch.arange(2*50).reshape(2,10,5)
print(x.shape)
input_indices = torch.tensor([[1], [2]])



print(x)

print(x[torch.arange(x.shape[0]).unsqueeze(-1), input_indices])

"""
it seems that it requires a 2d matrix to get a 2d tensor right ? 

2,3

soo expected shape here is [2, 3, 5]

but i don't get that hmm one way to get around this is to strait up have a very large input matrix 

soo yeah 

[[ 0.8419,  0.0608, -0.6204,  1.0084,  1.7739,  0.2415, -1.3456,  0.5532,
          0.8863, -0.7404],
[ 0.6352, -0.8284,  1.2984, -0.0147, -0.5947, -0.2497,  0.4707, -0.6801,
    0.6103, -1.3752]],

soo the weights do not seem to fucking update no matter what like wha the fuck 
    
w mean 0.0085592605
w var 0.99178195
b.mean 1.3411045e-08
b.var 0.043331224

"""


In [ ]:
import torch
import time

# Define the tensor
tensor = torch.tensor([[1, 2], [3, 4], [5, 6]])

# Repeat version
start_time = time.time()
repeated_tensor = tensor.unsqueeze(1).repeat(1, 3, 1)
random_tensor = torch.randn(3, tensor.size(1))
random_tensor_repeated = random_tensor.repeat(tensor.size(0), 1, 1)
concatenated_tensor_repeat = torch.cat((repeated_tensor, random_tensor_repeated), dim=2)
end_time = time.time()
print("Repeat version time:", end_time - start_time)

# Expand version
start_time = time.time()
repeated_tensor = tensor.unsqueeze(1).repeat(1, 3, 1)
print(repeated_tensor.shape)
random_tensor = torch.randn(3, tensor.size(1))
print(random_tensor.unsqueeze(0).expand(tensor.size(0), -1, -1).shape)
concatenated_tensor_expand = torch.cat((repeated_tensor, random_tensor.unsqueeze(0).expand(tensor.size(0), -1, -1)), dim=2)
end_time = time.time()
print("Expand version time:", end_time - start_time)

print(concatenated_tensor_expand)